> **SOLUTION VERSION.** Every exercise is completed. Stuck in the workshop? Copy what you need from here.

# Lab 3: The Agent Loop and Memory
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/solutions/03_agent_loop_and_memory.ipynb)

In Lab 2 we ran one turn by hand: *think, act, observe*. An **agent** simply repeats that turn in a
**loop** until the job is done:

```
   question
      |
      v
  +--------+     +-------+     +---------+
  | THINK  | --> |  ACT  | --> | OBSERVE |
  | (LLM)  |     | (tool)|     | (result)|
  +--------+     +-------+     +---------+
      ^                             |
      +-------- repeat -------------+
      |
      v
  final answer   (or stop after max_steps)
```

That loop, plus tools, plus memory, is the core of almost every agent framework.

### Setup
Run the cell below. The last line tells you which **brain** is answering:
`openai` or `gemini` (real AI in the cloud), `ollama` (AI on your own computer) or `mock`
(offline practice brain). All of them work for every exercise in this workshop.

In Colab, the first time a notebook reads your key, a pop-up asks for permission:
click **Grant access**. Every lab asks once.

In [ ]:
# SETUP: run this cell first. It is safe to run more than once.
import os, sys, subprocess
REPO_URL = "https://github.com/bmdayal/from-prompt-to-agent"

if os.path.isdir("../workshop"):        # running locally from notebooks/ or solutions/
    sys.path.insert(0, "..")
elif os.path.isdir("workshop"):         # running from the repo folder
    sys.path.insert(0, ".")
else:                                   # running in Google Colab: download the workshop files
    if not os.path.isdir("from-prompt-to-agent"):
        print("Downloading workshop files...")
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    sys.path.insert(0, "from-prompt-to-agent")

import json
from workshop import llm

PROVIDER = "auto"   # "auto", "openai", "gemini", "ollama" or "mock" (offline practice brain)
llm.configure(PROVIDER)

## Build the loop
Read the code below from top to bottom: it is the diagram above, written in Python.
Two lines are missing. Fill in **TODO 1** and **TODO 2** (the hints show exactly what to type).

*What if you skip them?* Run it anyway and watch: the agent never sees a result, so it repeats
itself until `max_steps` stops it. That is a real bug in real agents.

In [ ]:
from workshop.tools import TOOLS, run_tool
from workshop.agent import agent_system_prompt, show

SYSTEM = agent_system_prompt(TOOLS)

def my_agent(question, history=None, max_steps=6):
    messages = list(history or []) + [{"role": "user", "content": question}]

    for step in range(1, max_steps + 1):
        # 1. THINK: ask the model what to do next
        reply = llm.ask(messages, system=SYSTEM, json_mode=True)
        messages.append({"role": "assistant", "content": reply})
        decision = llm.parse_json(reply)
        if not decision or "action" not in decision:
            show(step, "warn", "Reply was not valid JSON, asking again")
            messages.append({"role": "user", "content": "Observation: reply with one valid JSON object."})
            continue
        show(step, "think", decision.get("thought", ""))

        # 2. DONE? If the model chose final_answer, we stop
        if decision["action"] == "final_answer":
            answer = decision.get("args", {}).get("answer", "")
            show(step, "answer", answer)
            return answer, messages

        # 3. ACT: run the tool the model chose
        show(step, "act", f'{decision["action"]}({decision.get("args", {})})')
        result = run_tool(TOOLS, decision["action"], decision.get("args", {}))

        # 4. OBSERVE: give the result back to the model
        observation = json.dumps(result, default=str)
        show(step, "observe", observation)
        messages.append({"role": "user", "content": "Observation: " + observation})

    return "I stopped because I reached the step limit.", messages

## Run your agent
**Predict:** how many steps will it take? Which tools will it use?

In [ ]:
answer, convo = my_agent("Where is the session about memory on Friday?")

Read the trace: the agent **searched**, noticed it needed a room, **looked up details**, and only
then **answered**. Nobody told it to use two tools. It decided step by step.

Try a question that needs a different tool:

In [ ]:
answer, convo = my_agent("Do S04 and S05 clash?")

## Memory
Short-term memory is simply **the conversation so far** (the `messages` list).
Watch what happens to a follow-up question **without** memory:

In [ ]:
answer, convo = my_agent("Which sessions about agents are on Thursday?")
print("\n--- follow-up WITHOUT memory ---")
answer2, _ = my_agent("What about Friday?")

"What about Friday?" is meaningless on its own: the agent does not know we were talking about agents.
Now pass the earlier conversation as `history`:

In [ ]:
print("--- follow-up WITH memory ---")
answer3, convo = my_agent("What about Friday?", history=convo)

**Memory is powerful but not free:**
- The history grows every turn, so each call gets slower and more expensive.
- It can carry private details forward. Decide what is worth remembering.
- **Long-term memory** (saved to a file or database between sessions) is the same idea, stored outside the conversation.

## Your turn
1. Ask your agent your own question about the program. Count the steps.
2. Change `max_steps` to `1` and ask a question that needs two tools. What happens?

In [ ]:
answer, convo = my_agent("Who is speaking at the session on hallucinations, and where is it?")
print("\n--- with max_steps=1 ---")
answer, _ = my_agent("Who is speaking at the session on hallucinations, and where is it?", max_steps=1)

## Key takeaways
- **Agent = LLM + tools + a loop.** The LLM decides, the loop repeats, tools act.
- `max_steps` is a safety budget. Every agent needs a way to stop.
- **Memory** is the message history. Pass it along and the agent can handle follow-ups.

**Next: Lab 4, A Team of Agents.** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/solutions/04_multi_agent.ipynb)

Click the badge to open the next lab in a new Colab tab, then run its setup cell first. (Each lab is self-contained, so you can close this one.)